# Tulika's GTEx isoforms that Tim did not rescore

Builds the list of (variant, isoform) pairs to send back to Tim for rescoring.

**Steps**
1. Start from the request sent to Tim (`mp2_request_for_tim.tsv`): 21,737 missing + 1,000 control variants.
2. Keep only the rows of Tulika's file whose variant is on that request (and that have a score).
3. Drop Tulika's rows whose isoform is **not** in the GTEx expression file.
4. Of what remains, keep the (variant, isoform) pairs Tim did **not** return a score for.
5. Write the result to `data/AI_generated/tulika_gtex_isoforms_not_rescored_by_tim.tsv`.

**Isoform matching.** Everywhere in this notebook, isoforms are compared on the Ensembl ID **without** the
version suffix (`ENST00000369519.4` → `ENST00000369519`), because Tulika's version numbers often differ from
the GTEx / Tim versions for the same isoform.

**Tulika's scores** are carried along only as a reference, to show what she had. They are not used as scores.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, GTeX_file

DATA        = Path(str(DATA_DIR))
REQUEST     = DATA / "AI_generated" / "mp2_request_for_tim.tsv"
TIM_RETURN  = DATA / "MutPred2_scores" / "MutPred2_scores_missing_and_control.tsv"
TULIKA_FILE = DATA / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"
OUT         = DATA / "AI_generated" / "tulika_gtex_isoforms_not_rescored_by_tim.tsv"

for f in (REQUEST, TIM_RETURN, TULIKA_FILE, GTeX_file):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")


def strip_version(ids):
    """ENST00000369519.4 -> ENST00000369519"""
    return ids.str.split(".").str[0]

## 2. Step 1: the request variants

`missing = True` marks the 21,737 variants SJ had no score for; `False` marks the 1,000 controls.

In [ ]:
request = pd.read_csv(REQUEST, sep="\t")
request["group"] = request["missing"].map({True: "missing", False: "control"})

print(f"request variants : {len(request)}  (duplicated variants: {request['Variant'].duplicated().sum()})")
print(request["group"].value_counts().to_string())
request.head()

## 3. Step 2: map Tulika's file onto the request variants

Tulika's file is one row per variant x isoform: `mutation` is the variant (same `chr_pos_REF/ALT` format as
the request), `Feature` is her versioned isoform, `mutpred` is her score.

Filters, in order:
- keep rows whose variant is on the request
- drop rows with no score
- drop exact duplicate rows (her file repeats many rows word for word)

In [ ]:
tulika_raw = pd.read_csv(TULIKA_FILE, sep="\t")
print(f"Tulika file              : {len(tulika_raw)} rows, {tulika_raw['mutation'].nunique()} variants")

tu = tulika_raw[tulika_raw["mutation"].isin(request["Variant"])]
print(f"  on the request         : {len(tu)} rows, {tu['mutation'].nunique()} variants")

tu = tu[tu["mutpred"].notna()]
print(f"  with a score           : {len(tu)} rows, {tu['mutation'].nunique()} variants")

n = len(tu)
tu = tu.drop_duplicates()
print(f"  exact duplicates removed: {n - len(tu)}  -> {len(tu)} rows")

tu = tu.rename(columns={"mutation": "Variant", "external_gene_name": "gene",
                        "Feature": "transcript_tulika", "mutpred": "MutPred2_tulika_reference"})
tu["isoform_id"] = strip_version(tu["transcript_tulika"])
tu = tu.merge(request[["Variant", "group", "sheets", "consequence"]], on="Variant", how="left")
assert tu["group"].notna().all()

print()
print("request variants Tulika has a score for:")
print(tu.drop_duplicates("Variant")["group"].value_counts().to_string())

Check: after removing exact duplicates, does any (variant, isoform) still appear more than once? If so, it
is the same pair with different values (e.g. two different scores). These rows are kept and flagged.

In [ ]:
dup = tu.duplicated(["Variant", "transcript_tulika"], keep=False)
tu["tulika_conflicting_scores"] = dup
print(f"(variant, isoform) pairs that appear with different values: "
      f"{tu.loc[dup, ['Variant', 'transcript_tulika']].drop_duplicates().shape[0]}  ({dup.sum()} rows)")
tu[dup].sort_values(["Variant", "transcript_tulika"])

## 4. Step 3: drop Tulika's isoforms that are not in GTEx

The GTEx expression file lists 158 isoforms twice: once normally and once with a `_PAR_Y` suffix (their copy
in the region shared by the X and Y chromosomes). The `_PAR_Y` copies are ignored, so each isoform ID maps to
exactly one GTEx version. That version (`transcript_gtex`) is kept in the output, because it's the version Tim's
run uses.

In [ ]:
gtex_ids = pd.read_csv(GTeX_file, sep="\t", usecols=["transcript_id"])["transcript_id"]
gtex_ids = gtex_ids[~gtex_ids.str.endswith("_PAR_Y")]
gtex = pd.DataFrame({"isoform_id": strip_version(gtex_ids), "transcript_gtex": gtex_ids})
assert not gtex["isoform_id"].duplicated().any()
print(f"GTEx isoforms: {len(gtex)}")

tu = tu.merge(gtex, on="isoform_id", how="left")
in_gtex = tu["transcript_gtex"].notna()

print(f"Tulika rows                  : {len(tu)}")
print(f"  isoform in GTEx  (kept)    : {in_gtex.sum()}")
print(f"  isoform NOT in GTEx (drop) : {(~in_gtex).sum()}  ({tu.loc[~in_gtex, 'isoform_id'].nunique()} distinct isoforms)")

lost = set(tu.loc[~in_gtex, "Variant"]) - set(tu.loc[in_gtex, "Variant"])
print(f"  variants left with no GTEx isoform at all: {len(lost)}")

tu_g = tu[in_gtex].copy()
print()
print("variants remaining, by group:")
print(tu_g.drop_duplicates("Variant")["group"].value_counts().to_string())

## 5. Step 4: which of these did Tim not rescore?

From Tim's return, only rows with an actual score count; a row with a blank score counts as not returned.
Tim's file uses `ID` for the variant.

For each remaining Tulika row:
- `tim_scored_no_isoform = True`: Tim returned no scored isoform at all for this variant
- `tim_scored_no_isoform = False`: Tim scored other isoforms of this variant, but not this one

In [ ]:
tim = pd.read_csv(TIM_RETURN, sep="\t")
print(f"Tim rows: {len(tim)}, blank score: {tim['MutPred2 score'].isna().sum()} (not counted)")
tim = tim[tim["MutPred2 score"].notna()]

tim_pairs    = set(zip(tim["ID"], strip_version(tim["transcript"])))
tim_variants = set(tim["ID"])

tu_g["tim_returned_isoform"] = [p in tim_pairs for p in zip(tu_g["Variant"], tu_g["isoform_id"])]
tu_g["tim_scored_no_isoform"] = ~tu_g["Variant"].isin(tim_variants)

print(f"Tulika GTEx rows                 : {len(tu_g)}")
print(f"  Tim returned the same isoform  : {tu_g['tim_returned_isoform'].sum()}")
print(f"  Tim did NOT return it          : {(~tu_g['tim_returned_isoform']).sum()}")

In [ ]:
gap = tu_g[~tu_g["tim_returned_isoform"]]

summary = (gap.groupby(["group", "tim_scored_no_isoform"])
              .agg(variants=("Variant", "nunique"), rows=("Variant", "size")))
summary

## 6. Step 5: write the file

One row per (variant, Tulika isoform) that Tim did not rescore. `transcript_gtex` is the GTEx-versioned ID to
send to Tim; `transcript_tulika` is the version as it appears in her file.

In [ ]:
out = (gap[["Variant", "group", "gene", "transcript_gtex", "transcript_tulika", "Substitution",
            "MutPred2_tulika_reference", "tim_scored_no_isoform", "tulika_conflicting_scores",
            "sheets", "consequence"]]
         .sort_values(["group", "Variant", "transcript_gtex"])
         .reset_index(drop=True))

out.to_csv(OUT, sep="\t", index=False)
print(f"wrote {len(out)} rows, {out['Variant'].nunique()} variants, "
      f"{out['transcript_gtex'].nunique()} isoforms -> {OUT}")
out.head(10)

## 7. Summary

In [ ]:
for g in ["missing", "control"]:
    s = out[out["group"] == g]
    v = s.drop_duplicates("Variant")
    print(f"{g:8s}: {len(s):>6} rows, {len(v):>5} variants  "
          f"({v['tim_scored_no_isoform'].sum()} with no Tim isoform at all, "
          f"{(~v['tim_scored_no_isoform']).sum()} partially returned)")